**Nama Anggota:**\
**1. Ammalya Khoirun Nizak (35031554036)**\
**2. Muhammad Rifky Ihwan (25031554241)**

# **FEATURE ENGGINEERING PART 1**




In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np
import re

**1. One Hot Encoding & Bag of Words (BoW)**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documents = ["timnas indonesia menang pertandingan",
             "pelatih mengatur strategi pertandingan dan taktik",
             "kemenangan timnas dan strategi pelatih"]

vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(documents)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,dan,indonesia,kemenangan,menang,mengatur,pelatih,pertandingan,strategi,taktik,timnas
0,0,1,0,1,0,0,1,0,0,1
1,1,0,0,0,1,1,1,1,1,0
2,1,0,1,0,0,1,0,1,0,1


In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,dan,indonesia,kemenangan,menang,mengatur,pelatih,pertandingan,strategi,taktik,timnas
0,0,1,0,1,0,0,1,0,0,1
1,1,0,0,0,1,1,1,1,1,0
2,1,0,1,0,0,1,0,1,0,1


**2. TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df

,dan,indonesia,kemenangan,menang,mengatur,pelatih,pertandingan,strategi,taktik,timnas
0,0.000000,0.562829,0.000000,0.562829,0.000000,0.000000,0.428046,0.000000,0.000000,0.428046
1,0.366180,0.000000,0.000000,0.000000,0.481482,0.366180,0.366180,0.366180,0.481482,0.000000
2,0.417796,0.000000,0.549351,0.000000,0.000000,0.417796,0.000000,0.417796,0.000000,0.417796


# Applied to the preprocess before
Sumber data : ANTARA News
kategori Sepak Bola Nasional

Data yang dipakai untuk feature engineering adalah teks yang sudah dibersihkan (huruf kecil, karakter yang tidak diperlukan dihapus, dan spasi dirapihkan)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

print("Mulai mengambil data dari ANTARA News...")

titles = []
base_url = "https://www.antaranews.com/slug/sepak-bola-nasional"

for page in range(1, 20):
    url = base_url if page == 1 else f"{base_url}/{page}"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers, timeout=20)
    if response.status_code != 200:
        print(f"Halaman {page} gagal: status {response.status_code}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    for h2 in soup.select("h2"):
        a = h2.find("a", href=True)
        if not a:
            continue

        href = urljoin(url, a["href"])
        title = a.get_text(" ", strip=True)

        if "/berita/" in href and len(title) > 15:
            titles.append(title)

raw_df = pd.DataFrame({"raw_title": titles}).drop_duplicates().reset_index(drop=True)

print(f"Total judul setelah scraping: {len(raw_df)}")
raw_df.head()


Mulai mengambil data dari ANTARA News...
Total judul setelah scraping: 285


,raw_title
0,Timnas Indonesia naik peringkat setelah juara ...
1,"Timnas Indonesia juara, Rizky Ridho: Alhamudli..."
2,Jay Idzes bangga melihat perjuangan Timnas Ind...
3,"Bawa jersi Marselino, Ole Romeny: Gelar ini ju..."
4,"Juara FIFA ASEAN Cup, Shayne Pattynama: Ini ba..."


In [ ]:
# Tahap preprocessing ringan
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df_scraped = raw_df.copy()
df_scraped["prepro"] = df_scraped["raw_title"].apply(preprocess_text)

df_scraped = df_scraped[df_scraped["prepro"].str.len() > 0].reset_index(drop=True)

df_scraped.to_csv("antara_sepakbola_preprocessed.csv", index=False)

print(f"Dataset preprocessing tersimpan: {len(df_scraped)} baris")
df_scraped.head()


Dataset preprocessing tersimpan: 285 baris


,raw_title,prepro
0,Timnas Indonesia naik peringkat setelah juara ...,timnas indonesia naik peringkat setelah juara ...
1,"Timnas Indonesia juara, Rizky Ridho: Alhamudli...",timnas indonesia juara rizky ridho alhamudli...
2,Jay Idzes bangga melihat perjuangan Timnas Ind...,jay idzes bangga melihat perjuangan timnas ind...
3,"Bawa jersi Marselino, Ole Romeny: Gelar ini ju...",bawa jersi marselino ole romeny gelar ini ju...
4,"Juara FIFA ASEAN Cup, Shayne Pattynama: Ini ba...",juara fifa asean cup shayne pattynama ini ba...


In [ ]:
df = pd.read_csv("antara_sepakbola_preprocessed.csv")
df = df.dropna(subset=["prepro"]).reset_index(drop=True)
df_pre = df["prepro"]

print("Jumlah dokumen:", len(df_pre))
df_pre.head()

Jumlah dokumen: 285


,prepro
0,timnas indonesia naik peringkat setelah juara ...
1,timnas indonesia juara rizky ridho alhamudli...
2,jay idzes bangga melihat perjuangan timnas ind...
3,bawa jersi marselino ole romeny gelar ini ju...
4,juara fifa asean cup shayne pattynama ini ba...


In [ ]:
# One Hot Encoding pada berita nyata
onehot_vectorizer = CountVectorizer(binary=True, min_df=2, max_features=2000)
X_onehot = onehot_vectorizer.fit_transform(df_pre)

df_onehot = pd.DataFrame(X_onehot.toarray(),
                         columns=onehot_vectorizer.get_feature_names_out())

print("Ukuran matriks One Hot:", df_onehot.shape)
df_onehot.head()


Ukuran matriks One Hot: (285, 337)


,000,10,17,2025,2026,2027,23,absen,ada,adalah,...,vickery,vietnam,villa,vs,walian,women,yakin,yang,yong,yordania
0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Bag of Words pada berita nyata
bow_vectorizer = CountVectorizer(min_df=2, max_features=2000)
X_bow = bow_vectorizer.fit_transform(df_pre)

df_bow = pd.DataFrame(X_bow.toarray(),
                      columns=bow_vectorizer.get_feature_names_out())

print("Ukuran matriks BoW:", df_bow.shape)
df_bow.head()


Ukuran matriks BoW: (285, 337)


,000,10,17,2025,2026,2027,23,absen,ada,adalah,...,vickery,vietnam,villa,vs,walian,women,yakin,yang,yong,yordania
0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# TF-IDF pada berita nyata
tfidf_vectorizer = TfidfVectorizer(min_df=2,
                                   max_features=2000,
                                   sublinear_tf=True)

X_tfidf = tfidf_vectorizer.fit_transform(df_pre)

df_FE = pd.DataFrame(X_tfidf.toarray(),
                     columns=tfidf_vectorizer.get_feature_names_out())

print("Ukuran matriks TF-IDF:", df_FE.shape)
df_FE.head()


Ukuran matriks TF-IDF: (285, 337)


,000,10,17,2025,2026,2027,23,absen,ada,adalah,...,vickery,vietnam,villa,vs,walian,women,yakin,yang,yong,yordania
0,0.0,0.0,0.0,0.0,0.226069,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Analisis Feature Engineering 1

One Hot Encoding bekerja dengan cara menandai apakah sebuah kata muncul atau tidak dalam suatu dokumen. Sementara itu, BoW (Bag of Words) menghitung seberapa sering kata tersebut muncul. Berbeda lagi dengan TF‑IDF, metode ini memberi nilai lebih tinggi pada kata yang dianggap penting dalam dokumen tertentu, dan nilai lebih rendah pada kata yang terlalu umum.

# **Feature Engineering Part 2 (Word Embeddings)**

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(str(news).lower())
             for news in tqdm(df_pre, desc="Tokenisasi")]

sentences[:2]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
Tokenisasi: 100%|██████████| 285/285 [00:00<00:00, 3984.51it/s]


[['timnas',
  'indonesia',
  'naik',
  'peringkat',
  'setelah',
  'juara',
  'fifa',
  'asean',
  'cup',
  '2026'],
 ['timnas',
  'indonesia',
  'juara',
  'rizky',
  'ridho',
  'alhamudlillah',
  'bukan',
  'mimpi',
  'lagi']]

In [ ]:
from nltk import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(str(news).lower()) for news in tqdm(df_pre)]
sentences[:2]

100%|██████████| 285/285 [00:00<00:00, 9222.52it/s]


[['timnas',
  'indonesia',
  'naik',
  'peringkat',
  'setelah',
  'juara',
  'fifa',
  'asean',
  'cup',
  '2026'],
 ['timnas',
  'indonesia',
  'juara',
  'rizky',
  'ridho',
  'alhamudlillah',
  'bukan',
  'mimpi',
  'lagi']]

# Build Word2Vec Model

In [ ]:
!pip install gensim

In [ ]:
import gensim.models

model = gensim.models.Word2Vec(sentences,
                               min_count=2,
                               vector_size=100,
                               window=5,
                               sg=0,
                               epochs=20,
                               workers=4,
                               seed=42)


param



*   min_count=2: hanya kata yang muncul minimal 2 kali.
*   vector_size=100: setiap kata direpresentasikan dalam 100 dimensi.
*   window=5: konteks kata diambil dari jendela 5 kata.
*   sg=0: menggunakan CBOW
*   epochs=20: model dilatih beberapa putaran agar representasi lebih stabil.
*   seed=42: hasil lebih konsisten saat notebook dijalankan ulang.

In [ ]:
model.save('sepakbola_antara.w2v')

In [ ]:
model = gensim.models.Word2Vec.load('sepakbola_antara.w2v')

In [ ]:
w2v = model.wv

print("10 kata teratas:")
print(w2v.index_to_key[:10])

print("\nUkuran vocabulary:", len(w2v))
print("Dimensi vektor:", w2v.vector_size)

10 kata teratas:
['indonesia', 'cup', 'timnas', '2026', 'piala', 'di', 'fifa', 'asean', 'putri', 'ke']

Ukuran vocabulary: 345
Dimensi vektor: 100


In [ ]:
# Uji kemiripan konteks kata
kata_uji = "timnas"

if kata_uji in w2v.key_to_index:
    w2v.similar_by_key(kata_uji, topn=10)
else:
    print(f"Kata '{kata_uji}' tidak ada di vocabulary.")

In [ ]:
# FE result
def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens
                     if word in model.wv]

    if not valid_vectors:
        return np.zeros(model.vector_size)

    return np.mean(valid_vectors, axis=0)

X_features_w2v = np.array([get_document_vector(doc, model)
                           for doc in sentences])

df_w2v = pd.DataFrame(X_features_w2v,
                      columns=[f"feature_{i+1}" for i in range(model.vector_size)])

print("Ukuran fitur dokumen Word2Vec:", df_w2v.shape)
df_w2v.head()


Ukuran fitur dokumen Word2Vec: (285, 100)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100
0,0.045709,0.051427,-0.072222,-0.022924,-0.040792,0.117655,0.086327,0.131706,-0.135572,0.014002,...,-0.210794,0.177177,0.078521,-0.151780,0.107503,-0.151990,0.038048,0.051328,-0.091054,-0.031941
1,0.043141,0.050757,-0.073898,-0.018024,-0.041108,0.119076,0.091066,0.138999,-0.134540,0.013542,...,-0.213642,0.182072,0.075136,-0.155061,0.107542,-0.153379,0.037777,0.054516,-0.087125,-0.037945
2,0.042093,0.048908,-0.068274,-0.020245,-0.038320,0.113000,0.087458,0.124172,-0.128614,0.010108,...,-0.206146,0.170050,0.074639,-0.148734,0.101782,-0.140376,0.033226,0.051656,-0.081253,-0.030956
3,0.026809,0.032128,-0.047596,-0.013659,-0.025473,0.071482,0.050718,0.083727,-0.088889,0.006888,...,-0.128550,0.109788,0.047845,-0.091920,0.067900,-0.093749,0.020638,0.028408,-0.049536,-0.021652
4,0.036888,0.035987,-0.054573,-0.016390,-0.029590,0.089071,0.065987,0.102561,-0.102000,0.013125,...,-0.161205,0.134474,0.059187,-0.110907,0.082136,-0.116185,0.030747,0.037532,-0.068803,-0.023686


**Visualize Word2Vec**

In [ ]:
!pip install umap-learn

In [ ]:
from umap import UMAP
import plotly.express as px

In [ ]:
TOP_N = min(100, len(w2v))
top_words = w2v.index_to_key[:TOP_N]
vectors_top = np.array([w2v[word] for word in top_words])

reducer = UMAP(n_neighbors=min(10, TOP_N - 1),
               min_dist=0.7,
               spread=1.5,
               metric="cosine",
               init="random",
               random_state=42)

X_2d = reducer.fit_transform(vectors_top)

df2 = pd.DataFrame(X_2d, columns=["umap1", "umap2"])
df2["word"] = top_words

fig = px.scatter(df2,
                 x="umap1",
                 y="umap2",
                 text="word",
                 hover_name="word")

fig.update_traces(textposition="top center",
                  marker=dict(size=7),
                  textfont=dict(size=9))

fig.update_layout(height=850,
                  title_text="Word2Vec Visualization - Berita Sepak Bola ANTARA",
                  template="plotly_white")

fig.show()

**Build FastText Model**

In [ ]:
model_fasttext = gensim.models.FastText(sentences,
                                        min_count=2,
                                        vector_size=100,
                                        window=5,
                                        min_n=3,
                                        max_n=6,
                                        sg=1,
                                        epochs=20,
                                        workers=4,
                                        seed=42)

parameter FastText



*   min_count=2: buang kata yang terlalu jarang muncul.
*   vector_size=100: tiap kata punya 100 angka fitur.
*   window=5: lihat konteks 5 kata di sekitar.
*   min_n=3, max_n=6: pakai potongan huruf 3-6 karakter.
*   sg=1: metode Skip‑Gram, tebak kata sekitar.
*   epochs=20: latih model 20 kali putaran.

In [ ]:
model_fasttext.save('sepakbola_antara.fasttext')

In [ ]:
model_fasttext = gensim.models.Word2Vec.load('sepakbola_antara.fasttext')

In [ ]:
fastext = model_fasttext.wv

print("10 kata teratas:")
print(fastext.index_to_key[:10])

print("\nUkuran vocabulary:", len(fastext))
print("Dimensi vektor:", fastext.vector_size)

10 kata teratas:
['indonesia', 'cup', 'timnas', '2026', 'piala', 'di', 'fifa', 'asean', 'putri', 'ke']

Ukuran vocabulary: 345
Dimensi vektor: 100


In [ ]:
# Uji kemiripan kata
kata_uji = "pelatih"

if kata_uji in fastext.key_to_index:
    fastext.similar_by_key(kata_uji, topn=10)
else:
    print(f"Kata '{kata_uji}' tidak ada di vocabulary.")

In [ ]:
# FE result
def get_document_vector_fasttext(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens
                     if word in model.wv]

    if not valid_vectors:
        return np.zeros(model.vector_size)

    return np.mean(valid_vectors, axis=0)

X_features_fasttext = np.array([get_document_vector_fasttext(doc, model_fasttext)
                                for doc in sentences])

df_fasttext = pd.DataFrame(X_features_fasttext,
                           columns=[f"feature_{i+1}" for i in range(model_fasttext.vector_size)])

print("Ukuran fitur dokumen FastText:", df_fasttext.shape)
df_fasttext.head()


Ukuran fitur dokumen FastText: (285, 100)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100
0,0.014516,-0.104139,0.006133,0.036290,0.283174,0.184161,0.043753,-0.180412,0.049957,-0.027737,...,-0.086957,-0.096218,0.260053,0.134142,0.004087,0.111999,0.002948,0.121315,-0.009398,-0.002395
1,0.012675,-0.089885,0.004153,0.032906,0.248071,0.162060,0.037327,-0.159034,0.044922,-0.026147,...,-0.076263,-0.084029,0.227014,0.118293,0.002711,0.097080,0.001859,0.107325,-0.006604,-0.002096
2,0.013387,-0.104589,0.006535,0.038421,0.287462,0.188515,0.041599,-0.184920,0.051818,-0.030467,...,-0.088255,-0.097993,0.262204,0.135615,0.002721,0.112820,0.003499,0.125376,-0.007376,-0.002038
3,0.012760,-0.092794,0.004992,0.033206,0.253790,0.166915,0.035535,-0.161749,0.046545,-0.026327,...,-0.077927,-0.086579,0.231463,0.119205,0.002798,0.098030,0.003863,0.110435,-0.006350,-0.000596
4,0.014286,-0.101252,0.005561,0.035702,0.275285,0.179570,0.041774,-0.174936,0.048789,-0.027019,...,-0.084917,-0.093207,0.252151,0.130659,0.004865,0.108166,0.003163,0.118072,-0.008563,-0.001650


# Visualize FastText

In [ ]:
# dengan vocabulary terbatas
TOP_N = min(100, len(fastext))
top_words = fastext.index_to_key[:TOP_N]
vectors_top = np.array([fastext[word] for word in top_words])

reducer = UMAP(n_neighbors=min(10, TOP_N - 1),
               min_dist=0.7,
               spread=1.5,
               metric="cosine",
               init="random",
               random_state=42)

X_2d = reducer.fit_transform(vectors_top)

df2 = pd.DataFrame(X_2d, columns=["umap1", "umap2"])
df2["word"] = top_words

fig = px.scatter(df2,
                 x="umap1",
                 y="umap2",
                 text="word",
                 hover_name="word")

fig.update_traces(textposition="top center",
                  marker=dict(size=7),
                  textfont=dict(size=9))

fig.update_layout(height=850,
                  title_text="FastText Visualization - Berita Sepak Bola ANTARA",
                  template="plotly_white")

fig.show()


Analisis Feature Engineering Part 2

Word2Vec dan FastText sama-sama mengubah kata menjadi bentuk angka berupa vektor. Bedanya, FastText tidak hanya melihat kata utuh, tapi juga memanfaatkan potongan huruf kecil (subword). Dengan cara ini, kata yang punya bentuk mirip atau imbuhan serupa bisa tetap dipahami lebih baik.

UMAP dipakai hanya sebagai alat bantu visualisasi. Titik-titik yang muncul di grafik menunjukkan kata-kata yang vektornya punya pola mirip di ruang embedding. Namun, posisi titik di gambar 2D hanya sekadar gambaran hubungan antar kata.